
# Test Plan mínimo ejecutable para prácticas de IA

Notebook basado en el documento **MJG_TestPlan_básicos_con código**.

Este notebook permite ejecutar, documentar y obtener evidencias de calidad
sobre proyectos sencillos de Inteligencia Artificial.



## 1. Objetivo

Aplicar un conjunto mínimo de pruebas de **calidad de datos, métricas,
calidad del modelo y explicabilidad** sobre proyectos prácticos de IA.



## 2. Preparación común del entorno

Carga de librerías, dataset y configuración básica del proyecto.


In [ ]:

# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cargar dataset (modificar nombre)
df = pd.read_csv('dataset.csv')

# Configuración del proyecto
TARGET = 'nombre_columna_objetivo'
TIPO_PROBLEMA = 'clasificación'  # o 'regresión'

print('Dataset cargado')
print('Dimensiones:', df.shape)



## 3. Requisito 1 – Calidad de los datos


In [ ]:

# DT-01 - Valores nulos
missing = df.isnull().mean().sort_values(ascending=False) * 100
print('% valores nulos por columna:')
print(missing)
print('
Columnas con más del 5% de nulos:')
print(missing[missing > 5])


In [ ]:

# DT-02 - Duplicados
duplicados = df.duplicated().sum()
print('Número de registros duplicados:', duplicados)


In [ ]:

# DT-03 - Tipos de datos
print(df.dtypes)
print(df.describe(include='all'))


In [ ]:

# DT-04 - Outliers (IQR)
numeric_cols = df.select_dtypes(include=np.number).columns
outliers = {}
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers[col] = df[(df[col] < lower) | (df[col] > upper)][col].count()

print(pd.Series(outliers).sort_values(ascending=False))


In [ ]:

# DT-05 - Balanceo del target
if TARGET in df.columns:
    print(df[TARGET].value_counts(normalize=True) * 100)
else:
    print('Target no encontrado')



## 4. Requisito 2 – Métricas del proyecto IA


In [ ]:

from sklearn.model_selection import train_test_split
X = df.drop(columns=[TARGET])
y = df[TARGET]
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if TIPO_PROBLEMA == 'clasificación' else None
)

print(X_train.shape, X_test.shape)


In [ ]:

# MT-02 - Métricas clasificación
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

if TIPO_PROBLEMA == 'clasificación':
    modelo_clf = RandomForestClassifier(random_state=42)
    modelo_clf.fit(X_train, y_train)
    y_pred = modelo_clf.predict(X_test)
    print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:

# MT-03 - Métricas regresión
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if TIPO_PROBLEMA == 'regresión':
    modelo_reg = RandomForestRegressor(random_state=42)
    modelo_reg.fit(X_train, y_train)
    y_pred = modelo_reg.predict(X_test)
    print('MAE:', mean_absolute_error(y_test, y_pred))
    print('RMSE:', mean_squared_error(y_test, y_pred, squared=False))
    print('R2:', r2_score(y_test, y_pred))



## 5. Requisito 3 – Calidad del modelo


In [ ]:

# MQ-01 - Overfitting
modelo = modelo_clf if TIPO_PROBLEMA == 'clasificación' else modelo_reg
print('Train score:', modelo.score(X_train, y_train))
print('Test score:', modelo.score(X_test, y_test))


In [ ]:

# MQ-02 - Validación cruzada
from sklearn.model_selection import cross_val_score
scoring = 'f1_macro' if TIPO_PROBLEMA == 'clasificación' else 'r2'
scores = cross_val_score(modelo, X, y, cv=5, scoring=scoring)
print('Scores:', scores)
print('Media:', scores.mean())
print('Desviación:', scores.std())



## 6. Requisito 4 – Explicabilidad


In [ ]:

# EX-01 - Importancia de variables
if hasattr(modelo, 'feature_importances_'):
    imp = pd.Series(modelo.feature_importances_, index=X_train.columns)
    imp.sort_values(ascending=False).head(10).plot(kind='barh')
    plt.title('Importancia de variables')
    plt.gca().invert_yaxis()
    plt.show()
